In [12]:
import warnings
from jpholiday import JPHoliday


warnings.simplefilter('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import japanize_matplotlib
import matplotlib_fontja
%matplotlib inline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as MSE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import Normalizer

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train_add = pd.read_csv('train_add.csv')
add_2014 = pd.read_csv('2014_add.csv')
stadium = pd.read_csv('stadium.csv')
condition = pd.read_csv('condition.csv')
condition_add = pd.read_csv('condition_add.csv')

print('train shape: ', train.shape)
print('train_add shape: ', train_add.shape)
print('test shape: ',test.shape)
print('condition shape: ', condition.shape)
print('condition_add shape: ', condition_add.shape)
print('stadium shape:', stadium.shape)

full_train = pd.concat([train, train_add], axis=0)
full_condition = pd.concat([condition, condition_add], axis=0)

print('train concat')
print('before: ', train.shape, train_add.shape)
print('after: ', full_train.shape)

print('condition concat')
print('before: ', condition.shape, condition_add.shape)
print('after: ', full_condition.shape)

stadium = stadium.rename(columns={'name': 'stadium'})
stadium = stadium[['stadium', 'capa']]
full_condition = full_condition[['id',"home_score","away_score", 'weather' ,'temperature' ,'humidity']]

full_train = pd.merge(full_train, stadium, on='stadium',  how='left')
full_train = pd.merge(full_train, full_condition, on='id',  how='left')
full_test = pd.merge(test, stadium, on='stadium',  how='left')
full_test = pd.merge(full_test, full_condition, on='id',  how='left')

full_train.loc[full_train['stage'] == 'Ｊ１', 'stage'] = 'J1'
full_train.loc[full_train['stage'] == 'Ｊ２', 'stage'] = 'J2'

full_test.loc[full_test['stage'] == 'Ｊ１', 'stage'] = 'J1'
full_test.loc[full_test['stage'] == 'Ｊ２', 'stage'] = 'J2'

full_train['section'] = full_train['match'].apply(lambda x: x.split('節')[0][1:]).astype(int)
full_test['section'] = full_test['match'].apply(lambda x: x.split('節')[0][1:]).astype(int)


train shape:  (1721, 11)
train_add shape:  (232, 11)
test shape:  (313, 10)
condition shape:  (2034, 31)
condition_add shape:  (270, 31)
stadium shape: (59, 3)
train concat
before:  (1721, 11) (232, 11)
after:  (1953, 11)
condition concat
before:  (2034, 31) (270, 31)
after:  (2304, 31)


In [13]:
full_train['capa']

0       19694
1       40000
2       21000
3       50000
4       39232
        ...  
1948    12000
1949    15576
1950    15454
1951    12000
1952    15135
Name: capa, Length: 1953, dtype: int64

In [16]:

def cal_score(row):
    if row['home_score'] ==  row['away_score']:
        row['sum_score_' + row["home"]] = 1 + row['home_score'] * 0.001
        row['sum_score_' + row["away"]] = 1 + row['away_score'] * 0.001
    elif row['home_score'] >  row['away_score']:
        row['sum_score_' + row['home']] = 3 + row['home_score'] * 0.001
        row['sum_score_' + row['away']] = 0 + row['away_score']* 0.001
    else:
        row['sum_score_' + row['away']] = 3 + row['away_score'] * 0.001
        row['sum_score_' + row['home']] = 0 + row['home_score']* 0.001
    return row

def calc_rank(df):
    all_teams_name = df['home'].unique().tolist()
    all_teams_sum_score = ['sum_score_' + x for x in all_teams_name]
    scores = all_teams_sum_score
    tmp_columns = df.columns.tolist() + scores
    
    
    
    df_scores = df.apply(cal_score, axis=1)
    print(f"{df_scores=}")
    df_scores = df_scores.fillna(0)
    df_scores = df_scores.sort_values(['year', 'section'])
    
    
    df_scores_2012 = df_scores[df_scores['year'] == 2012]
    df_scores_2013 = df_scores[df_scores['year'] == 2013]
    df_scores_2014 = df_scores[df_scores['year'] == 2014]
    
    
    df_scores_2012[scores] = df_scores_2012[scores].cumsum()
    df_scores_2013[scores] = df_scores_2013[scores].cumsum()
    df_scores_2014[scores] = df_scores_2014[scores].cumsum()
    
    df_scores = pd.concat([df_scores_2012, df_scores_2013, df_scores_2014])
    df_scores = df_scores.sort_values(['year', 'section'])
    
    
    df_scores_J1 = df_scores[df_scores['stage'] == 'J1']
    df_scores_J2 = df_scores[df_scores['stage'] == 'J2']
    
    df_scores_J1 = df_scores_J1.sort_values(['year', 'section'])
    df_scores_J2 = df_scores_J2.sort_values(['year', 'section'])
    
    names_J1_base = df_scores_J1['home'].dropna().unique().tolist()
    names_J2_base = df_scores_J2['home'].dropna().unique().tolist()
    names_J1 = ['sum_score_' + x for x in names_J1_base]
    names_J2 = ['sum_score_' + x for x in names_J2_base]
    
    rank_J1_df = df_scores_J1[names_J1].rank(axis=1, ascending=False, method='min')
    rank_J2_df = df_scores_J2[names_J2].rank(axis=1, ascending=False, method='min')
    
    df_scores_J1 = df_scores_J1.drop(names_J1, axis=1)
    df_scores_J2 = df_scores_J2.drop(names_J2, axis=1)
    
    
    con_rank_J1 = pd.concat([df_scores_J1, rank_J1_df],axis=1)
    con_rank_J2 = pd.concat([df_scores_J2, rank_J2_df],axis=1)
    
    teams_match = ['year', 'section'] + scores
    
    melted_J1 = con_rank_J1.melt(value_vars=['home', 'away','stage'], value_name='team',id_vars=names_J1 + ['year', 'section', 'stage'])
    melted_J2 = con_rank_J2.melt(value_vars=['home', 'away','stage'], value_name='team',id_vars=names_J2 + ['year', 'section', 'stage'])
    
    def calc_win_rate(melted, names):
        melted['win_rate'] = 0
        melted = melted.sort_values(['year','section'])
        for name in names:
            melted.loc[melted['team'] == name, 'win_rate'] = melted['sum_score_' + name]
            melted.loc[melted['team'] == name, 'win_rate'] = melted.loc[melted['team'] == name]['win_rate'].shift(1)
            melted.loc[melted['win_rate'].isnull(), 'win_rate'] = 100
        return melted
    melted_J1 = calc_win_rate(melted_J1, names_J1_base)
    melted_J2 = calc_win_rate(melted_J2, names_J2_base)
    
    
    
    standing_J1 = melted_J1[["year","section","stage","team","win_rate"]]
    standing_J2 = melted_J2[["year","section","stage","team","win_rate"]]
    standing_df = pd.concat([standing_J1, standing_J2])
    #standing_df['win_rate2'] = standing_df['win_rate']
    
    df['home_rank'] = 0
    df['away_rank'] = 0
    
    pd.merge(df, standing_df, left_on=['year', 'section', 'stage', 'away'], right_on=['year', 'section', 'stage', 'team'])
    df_home = pd.merge(df, standing_df, left_on=['year', 'section', 'stage', 'home'], right_on=['year', 'section', 'stage', 'team'], how='left')
    df_away = pd.merge(df, standing_df, left_on=['year', 'section', 'stage', 'away'], right_on=['year', 'section', 'stage', 'team'], how='left')
    df['home_rank'] = df_home['win_rate']
    df['away_rank'] = df_away['win_rate']

#    df.loc[df['stage'] == 'J1', "home_rank_section"] = df['home_rank'] / df['section'] * 34
    #df.loc[df['stage'] == 'J1', "away_rank_section"] = df['away_rank'] / df['section'] * 34
    #df.loc[df['stage'] == 'J2', "home_rank_section"] = df['home_rank'] / df['section'] * 42
    #df.loc[df['stage'] == 'J2', "away_rank_section"] = df['away_rank'] / df['section'] * 42
    return df

In [17]:
full_train = calc_rank(full_train)
full_test = calc_rank(full_test)
#full_train_rank.groupby(['year','stage'])['section'].max()

df_scores=           away  away_rank  away_score   capa   gameday       home  home_rank  \
0      鹿島アントラーズ      100.0           0  19694  03/10(土)     ベガルタ仙台      100.0   
1       清水エスパルス      100.0           0  40000  03/10(土)   名古屋グランパス      100.0   
2       ヴィッセル神戸      100.0           3  21000  03/10(土)      ガンバ大阪      100.0   
3         浦和レッズ      100.0           0  50000  03/10(土)  サンフレッチェ広島      100.0   
4        ジュビロ磐田      100.0           0  39232  03/10(土)   コンサドーレ札幌      100.0   
...         ...        ...         ...    ...       ...        ...        ...   
1948   ザスパクサツ群馬       20.0           0  12000  07/20(日)  水戸ホーリーホック       12.0   
1949  京都サンガF.C.        8.0           0  15576  07/20(日)       愛媛ＦＣ       15.0   
1950     ジュビロ磐田        3.0           0  15454  07/26(土)       横浜ＦＣ       21.0   
1951  Ｖ・ファーレン長崎       15.0           0  12000  07/30(水)  水戸ホーリーホック       13.0   
1952  ギラヴァンツ北九州        4.0           0  15135  07/30(水)   ザスパクサツ群馬       16.0   

      home_score 

In [5]:
##### full_train['day'] = full_train['gameday'].apply(lambda x: x[:4]).astype(int)
full_train['month'] = full_train['gameday'].apply(lambda x: x[:2]).astype(int)
full_train['weekday'] = full_train['gameday'].apply(lambda x: x[6])
full_test['month'] = full_test['gameday'].apply(lambda x: x[:2]).astype(int)
full_test['weekday'] = full_test['gameday'].apply(lambda x: x[6])

full_train['holiday'] = False
full_test['holiday'] = False
full_train.loc[(full_train['gameday'].str.contains('祝')) | (full_train['gameday'].str.contains('休')), 'holiday'] = True
full_test.loc[(full_test['gameday'].str.contains('祝')) | (full_test['gameday'].str.contains('休')), 'holiday'] = True


# timeから時間を取り出す
full_train['hour'] = full_train['time'].apply(lambda x: x.split(':')[0]).astype(int)
full_test['hour'] = full_test['time'].apply(lambda x: x.split(':')[0]).astype(int)

bins = [0, 13, 17, 18, 24]
labels = ['hour1', 'hour2', 'hour3', 'hour4']
full_train['hour'] = pd.cut(full_train['hour'], bins=bins, labels=labels, right=True)
full_test['hour']= pd.cut(full_test['hour'], bins=bins, labels=labels, right=True)


# tvからサービスの数をカウント
full_train['num_tv'] = full_train['tv'].apply(lambda x: len(x.split('／')))
full_test['num_tv'] = full_test['tv'].apply(lambda x: len(x.split('／')))

full_train['nhk'] = 'none'
full_train.loc[(full_train['tv'].str.contains('ＮＨＫ')), 'nhk'] = "nhk1"
full_train.loc[(full_train['tv'].str.contains('ＮＨＫ総合')), 'nhk'] = "nhk2"

full_test['nhk'] = 'none'
full_test.loc[(full_test['tv'].str.contains('ＮＨＫ')), 'nhk'] = "nhk1"
full_test.loc[(full_test['tv'].str.contains('ＮＨＫ総合')), 'nhk'] = "nhk2"


full_train['other_tv'] = False
full_train.loc[(full_train['tv'].str.contains('テレ')) | (full_train['tv'].str.contains('ＴＯＫＹＯ'))| (full_train['tv'].str.contains('放送')), 'other_tv'] = True
full_test['other_tv'] = False
full_test.loc[(full_test['tv'].str.contains('テレ')) | (full_test['tv'].str.contains('ＴＯＫＹＯ'))| (full_test['tv'].str.contains('放送')), 'other_tv'] = True

full_train['rain'] = False
full_train.loc[(full_train['weather'].str.contains('雨')) | (full_train['weather'].str.contains('雪')), 'rain'] = True
full_test['rain'] = False
full_test.loc[(full_test['weather'].str.contains('雨')) | (full_test['weather'].str.contains('雪')), 'rain'] = True

full_train['first'] = False
full_train['second'] = False
full_train['last'] = False
full_train.loc[(full_train['section'] >= 1) & (full_train['section'] <= 2), "first"] = True
full_train.loc[(full_train['second'] >= 3) & (full_train['section'] <= 7), "second"] = True
full_train.loc[(full_train['stage'] == "J1") & (full_train['section'] >= 33), "last"] = True
full_train.loc[(full_train['stage'] == "J2") & (full_train['section'] >= 41), "last"] = True

full_test['first'] = False
full_test['second'] = False
full_test['last'] = False
full_test.loc[(full_test['section'] >= 1) & (full_test['section'] <= 2), "first"] = True
full_test.loc[(full_test['second'] >= 3) & (full_test['section'] <= 7), "second"] = True
full_test.loc[(full_test['stage'] == "J1") & (full_test['section'] >= 33), "last"] = True
full_test.loc[(full_test['stage'] == "J2") & (full_test['section'] >= 41), "last"] = True


In [6]:
# humidityから％を切り取り数値データに変換
#full_train['humidity'] = full_train['humidity'].apply(lambda x: x.rstrip('%')).astype(int)
#full_test['humidity'] = full_test['humidity'].apply(lambda x: x.rstrip('%')).astype(int)
full_test[(full_test['home_rank'] < 90) & (full_test['stage'] == 'J1')]['home_rank'].max()
full_test[(full_test['home_rank'] < 91) & (full_test['stage'] == 'J2')]['home_rank'].max()
#full_test['home_rank']
#full_train[full_train['home'] == '']

np.float64(22.0)

In [7]:
full_train['rain'].unique()
full_train.iloc[30:70, 15:]


,temperature,humidity,section,home_rank,away_rank,month,weekday,holiday,hour,num_tv,nhk,other_tv,rain,first,second,last
30,10.9,47%,4,15.0,19.0,3,土,False,hour3,3,none,False,False,False,False,False
31,9.4,50%,4,17.0,22.0,3,土,False,hour4,3,none,False,True,False,False,False
32,10.1,25%,4,10.0,4.0,3,土,False,hour4,4,nhk1,False,False,False,False,False
33,20.8,32%,5,21.0,13.0,4,土,False,hour2,4,nhk1,False,False,False,False,False
34,6.1,53%,5,1.0,1.0,4,土,False,hour2,5,nhk1,False,False,False,False,False
35,9.6,26%,5,5.0,8.0,4,土,False,hour2,5,nhk1,False,False,False,False,False
36,12.0,28%,5,10.0,11.0,4,土,False,hour2,3,none,False,False,False,False,False
37,3.4,63%,5,19.0,18.0,4,土,False,hour2,4,nhk1,False,False,False,False,False
38,9.8,20%,5,14.0,8.0,4,土,False,hour2,3,none,False,False,False,False,False
39,9.3,31%,5,22.0,11.0,4,土,False,hour2,3,none,False,False,False,False,False


In [8]:
label = full_train['y']
#full_train = full_train.drop(['id', 'gameday', 'time', 'stadium', 'tv', 'weather', 'temperature', 'humidity', 'home_score','away_score', 'month', 'match','y', 'section', 'num_tv'],axis=1)
full_train = full_train.drop(['id', 'gameday', 'time', 'stadium', 'tv', 'weather', 'temperature', 'humidity', 'home_score','away_score', 'month', 'match',  'num_tv'],axis=1)
full_test= full_test.drop(['id', 'gameday', 'time', 'stadium', 'tv', 'weather', 'temperature', 'humidity', 'home_score','away_score', 'month','match','section', 'num_tv'],axis=1)

In [9]:
full_train.columns

Index(['y', 'year', 'stage', 'home', 'away', 'capa', 'section', 'home_rank',
       'away_rank', 'weekday', 'holiday', 'hour', 'nhk', 'other_tv', 'rain',
       'first', 'second', 'last'],
      dtype='str')

In [10]:
full_test.columns
set(full_test.columns)-set(drop_col)
full_train.info()

NameError: name 'drop_col' is not defined

In [11]:
from sklearn.preprocessing import OneHotEncoder

sum_df = pd.concat([full_train, full_test], axis=0)
#drop_col = ["section",'capa','home_rank', 'away_rank', 'num_tv']
drop_col = ['capa','home_rank', 'away_rank']
#sum_df = sum_df.drop(drop_col, axis=1)

numerical_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

numerical_features = ['capa', 'home_rank', 'away_rank']
categorical_features = list(set(full_train.columns) - set(numerical_features))

numerical_transformer.fit(full_train[drop_col])
categorical_transformer.fit(full_train[categorical_features])


col_transformers = ColumnTransformer(
    transformers=[
        # (名前, 変換器, 対象列) のタプルで指定
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    # 指定していない列をどう扱うか。ここでは除外（デフォルト）。
    # もし残したい列があれば remainder='passthrough' を指定。
    remainder='passthrough'
)
col_transformers.fit(full_train)

col_transformers.set_output(transform='pandas')
train_df = col_transformers.transform(full_train)
test_df = col_transformers.transform(full_test)


##ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#ohe.fit(sum_df)
#feature_names= ohe.get_feature_names_out(input_features=sum_df.columns)
#
#full_train_num = full_train[drop_col]
#train_oneshot = ohe.transform(full_train.drop(drop_col,axis=1))
#train_oneshot_df = pd.DataFrame(train_oneshot, columns=feature_names)
#
#full_test_num = full_test[drop_col]
#test_oneshot = ohe.transform(full_test.drop(drop_col,axis=1))
#test_oneshot_df = pd.DataFrame(test_oneshot, columns=feature_names)
#

ValueError: columns are missing: {'y', 'section'}

In [12]:
train_df.columns[0:20]
test_df.isnull().sum().sum()

NameError: name 'test_df' is not defined

In [13]:
print(full_train.shape)
full_train.columns
#feature_names= ohe.get_feature_names_out(input_features=sum_df.columns)

(1953, 18)


Index(['y', 'year', 'stage', 'home', 'away', 'capa', 'section', 'home_rank',
       'away_rank', 'weekday', 'holiday', 'hour', 'nhk', 'other_tv', 'rain',
       'first', 'second', 'last'],
      dtype='object')

In [14]:
encoded_df = pd.DataFrame(onehot_df, columns=feature_names)

NameError: name 'onehot_df' is not defined

In [15]:
encoded_df.columns

NameError: name 'encoded_df' is not defined

In [16]:
numerical_features 
full_train[full_train['year'] == 2014]

,y,year,stage,home,away,capa,section,home_rank,away_rank,weekday,holiday,hour,nhk,other_tv,rain,first,second,last
1357,15852,2014,J1,ベガルタ仙台,アルビレックス新潟,19694,1,17.0,10.0,土,False,hour2,nhk1,False,False,True,False,False
1358,13809,2014,J1,ヴァンフォーレ甲府,鹿島アントラーズ,54224,1,19.0,5.0,土,False,hour2,nhk1,False,False,True,False,False
1359,37079,2014,J1,セレッソ大阪,サンフレッチェ広島,47816,1,6.0,3.0,土,False,hour2,nhk2,False,True,True,False,False
1360,14623,2014,J1,柏レイソル,ＦＣ東京,15349,1,13.0,10.0,土,False,hour2,none,False,True,True,False,False
1361,18438,2014,J1,ガンバ大阪,浦和レッズ,21000,1,20.0,7.0,土,False,hour4,nhk1,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,5776,2014,J2,水戸ホーリーホック,ザスパクサツ群馬,12000,22,12.0,20.0,日,False,hour3,none,False,False,False,False,False
1949,3865,2014,J2,愛媛ＦＣ,京都サンガF.C.,15576,22,15.0,8.0,日,False,hour4,none,False,False,False,False,False
1950,6420,2014,J2,横浜ＦＣ,ジュビロ磐田,15454,23,21.0,3.0,土,False,hour3,none,False,False,False,False,False
1951,3711,2014,J2,水戸ホーリーホック,Ｖ・ファーレン長崎,12000,24,13.0,15.0,水,False,hour4,none,False,False,False,False,False


In [17]:
full_train[numerical_features]
full_test

,year,stage,home,away,capa,home_rank,away_rank,weekday,holiday,hour,nhk,other_tv,rain,first,second,last
0,2014,J1,ベガルタ仙台,大宮アルディージャ,19694,100.0,100.0,土,False,hour4,none,False,False,False,False,False
1,2014,J1,鹿島アントラーズ,サンフレッチェ広島,40728,100.0,100.0,土,False,hour3,none,False,False,False,False,False
2,2014,J1,浦和レッズ,ヴィッセル神戸,63700,100.0,100.0,土,False,hour4,nhk1,True,False,False,False,False
3,2014,J1,柏レイソル,川崎フロンターレ,15349,100.0,100.0,土,False,hour4,none,False,False,False,False,False
4,2014,J1,アルビレックス新潟,セレッソ大阪,42300,100.0,100.0,土,False,hour4,none,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308,2014,J2,カターレ富山,ファジアーノ岡山,25250,22.0,13.0,日,True,hour2,none,False,False,False,False,True
309,2014,J2,京都サンガF.C.,ＦＣ岐阜,20588,9.0,17.0,日,True,hour2,nhk1,False,False,False,False,True
310,2014,J2,カマタマーレ讃岐,ジェフユナイテッド千葉,22338,14.0,3.0,日,True,hour2,none,False,False,False,False,True
311,2014,J2,アビスパ福岡,ロアッソ熊本,22563,16.0,6.0,日,True,hour2,none,False,False,False,False,True


In [18]:
full_train['ratio_capa'] = full_train['y'] / full_train['capa'] * 100

In [19]:
#full_train.sort_values('ratio_capa')[-20:]
#full_train[full_train['home'] == 'ベガルタ仙台']

In [20]:
full_train2 = full_train[full_train['year'] <= 2014]

In [21]:
full_train2_train = full_train[((full_train['stage'] == 'J1') & (full_train['section'] <=17)) | ((full_train['stage'] == 'J2') & (full_train['section'] <=24))]
full_train2_test = full_train[((full_train['stage'] == 'J1') & (full_train['section'] >17)) | ((full_train['stage'] == 'J2') & (full_train['section'] >24))]

In [22]:
full_train2_train_2012 = full_train2_train[full_train2_train['year'] == 2012]
full_train2_train_2013 = full_train2_train[full_train2_train['year'] == 2013]
#full_train2_train_2014 = full_train2_train[full_train2_train['year'] == 2014]
full_train2_train_2014 = full_train[full_train['year'] == 2014]
full_train2_train_2012_grouped = full_train2_train_2012.groupby('home')['y']
full_train2_train_2013_grouped = full_train2_train_2013.groupby('home')['y']
full_train2_train_2014_grouped = full_train2_train_2014.groupby('home')['y']

#full_test_grouped = full_test.groupby('home')['y']

full_test['mean'] = full_test['home'].apply(lambda x: full_train2_train_2014_grouped.get_group(x).mean())


"""

full_train2_test_2012 = full_train2_test[full_train2_test['year'] == 2012]
full_train2_test_2013 = full_train2_test[full_train2_test['year'] == 2013]
full_train2_test_2014 = full_train2_test[full_train2_test['year'] == 2014]

full_train_2012 = full_train[full_train['year'] == 2012]
full_train_2013 = full_train[full_train['year'] == 2013]
full_train_2014 = full_train[full_train['year'] == 2014]
#full_train2_test.loc[full_train2_test['year'] == 2012, '']
#print(full_train2_train.groupby('home')['y'].mean())
#print(full_train2_test.groupby('home')['y'].mean())
full_train2_test_2012['mean'] = full_train2_test_2012['home'].apply(lambda x: full_train2_train_2012_grouped.get_group(x).mean())
full_train2_test_2013['mean'] = full_train2_test_2013['home'].apply(lambda x: full_train2_train_2013_grouped.get_group(x).mean())
full_train2_test_2014['mean'] = full_train2_test_2014['home'].apply(lambda x: full_train2_train_2014_grouped.get_group(x).mean())
"""

"\n\nfull_train2_test_2012 = full_train2_test[full_train2_test['year'] == 2012]\nfull_train2_test_2013 = full_train2_test[full_train2_test['year'] == 2013]\nfull_train2_test_2014 = full_train2_test[full_train2_test['year'] == 2014]\n\nfull_train_2012 = full_train[full_train['year'] == 2012]\nfull_train_2013 = full_train[full_train['year'] == 2013]\nfull_train_2014 = full_train[full_train['year'] == 2014]\n#full_train2_test.loc[full_train2_test['year'] == 2012, '']\n#print(full_train2_train.groupby('home')['y'].mean())\n#print(full_train2_test.groupby('home')['y'].mean())\nfull_train2_test_2012['mean'] = full_train2_test_2012['home'].apply(lambda x: full_train2_train_2012_grouped.get_group(x).mean())\nfull_train2_test_2013['mean'] = full_train2_test_2013['home'].apply(lambda x: full_train2_train_2013_grouped.get_group(x).mean())\nfull_train2_test_2014['mean'] = full_train2_test_2014['home'].apply(lambda x: full_train2_train_2014_grouped.get_group(x).mean())\n"

In [23]:
#full_train2_test_2012['mean']
#full_train2 = pd.concat([full_train2_test_2012, full_train2_test_2013, full_train2_test_2014], axis=0)
full_train2
#print(full_test.iloc[129:145])
full_train2_train_2014['home'].unique()
#full_train2_train_2014_grouped.sum()

print(full_test['mean'])
#print(full_train2_test.iloc[129:140])
#print(full_test['home'])
#print(full_test)
#print(len(full_train2_train_2014['home'].unique()))
#print((full_test['home'].unique()))
#full_train2_train_2014_grouped.get_group("ガイナーレ鳥取")
#l_train2_train_2014_grouped.get_group(x).mean())
#full_rain2_train.groupby('home')['y'].mean() / full_train2_test.groupby('home')['y'].mean()

0      13742.375000
1      15452.500000
2      34552.444444
3      11847.222222
4      23411.500000
           ...     
308     4546.416667
309     7576.363636
310     3361.909091
311     5247.500000
312     7473.250000
Name: mean, Length: 313, dtype: float64


In [24]:
#full_train2_train[full_train2_test['home'] == "ＦＣ町田ゼルビア"]['home_rank']

In [25]:
#len(full_train[full_train['stage'] == 'J2']['home'].unique())
full_train2_train.groupby('home')['y'].get_group("ＦＣ町田ゼルビア").mean()
grouped = full_train2_train.groupby('home')['y']

In [26]:
full_train2_train['mean'] = 0
#full_train2_train.loc[full_train2_train['home'] == ]
#full_train2_train['mean'] = full_train2_train.apply(lambda x: grouped.get_group("ＦＣ町田ゼルビア").mean())
full_train2_train[['count' , 'mean', 'std', 'min', 't25', 't50', 't75', 'max']] = full_train2_train['home'].apply(lambda x: grouped.get_group(x).describe())
#full_train['hour'] = full_train['time'].apply(lmbda x: x.split(':')[0]).astype(int)
#full_train2_train.iloc[1]['home']
grouped.get_group("ＦＣ町田ゼルビア").mean()

np.float64(4031.0833333333335)

In [27]:
full_train2_train['mean']

0       14473.160000
1       17000.800000
2       13128.000000
3       14772.777778
4       10791.363636
            ...     
1948     4398.388889
1949     3820.594595
1950     5384.114286
1951     4398.388889
1952     3358.227273
Name: mean, Length: 1251, dtype: float64

In [28]:
grouped.get_group('ＦＣ町田ゼルビア').describe()

count      12.000000
mean     4031.083333
std      1226.371474
min      2060.000000
25%      3252.000000
50%      3770.500000
75%      4394.500000
max      6288.000000
Name: y, dtype: float64